# Customer Support Router | Routing

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class RouterState(TypedDict):
    input: str
    category: str
    response: str

In [5]:
# Router node: classify the input and route using Command
def classify_input(state: RouterState) -> Command[Literal["billing_agent", "technical_agent", "general_agent"]]:
    response = model.invoke(
        f"Classify this query into exactly one category: 'billing', 'technical', or 'general'.\n"
        f"Query: {state['input']}\n"
        f"Respond with only the category name."
    )
    category = response.content.strip().lower()
    routing_map = {
        "billing": "billing_agent",
        "technical": "technical_agent",
        "general": "general_agent",
    }
    destination = routing_map.get(category, "general_agent")
    return Command(goto=destination, update={"category": category})

# Specialized handlers
def billing_agent(state: RouterState) -> dict:
    response = model.invoke(
        f"You are a billing specialist. Help with: {state['input']}"
    )
    return {"response": response.content}

def technical_agent(state: RouterState) -> dict:
    response = model.invoke(
        f"You are a technical support expert. Help with: {state['input']}"
    )
    return {"response": response.content}

def general_agent(state: RouterState) -> dict:
    response = model.invoke(
        f"You are a helpful assistant. Help with: {state['input']}"
    )
    return {"response": response.content}

In [6]:
# Build the graph
graph = StateGraph(RouterState)
graph.add_node("classifier", classify_input)
graph.add_node("billing_agent", billing_agent)
graph.add_node("technical_agent", technical_agent)
graph.add_node("general_agent", general_agent)

graph.add_edge(START, "classifier")
# No add_conditional_edges needed -- classify_input returns Command to route directly
graph.add_edge("billing_agent", END)
graph.add_edge("technical_agent", END)
graph.add_edge("general_agent", END)

router = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(router)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classifier(classifier)
	billing_agent(billing_agent)
	technical_agent(technical_agent)
	general_agent(general_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> classifier;
	classifier -.-> billing_agent;
	classifier -.-> general_agent;
	classifier -.-> technical_agent;
	billing_agent --> __end__;
	general_agent --> __end__;
	technical_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = router.invoke({"input": "I was charged twice on my last invoice"})
print(result["response"])

If you were charged twice on your last invoice, you’ll want to address this issue promptly to ensure a refund or adjustment. Here are the steps you can take to resolve the double charge:

1. **Review the Invoice**: Double-check the invoice details to confirm that there are indeed two charges for the same item or service, and ensure it’s not a misunderstanding or legitimate charge.

2. **Check Payment Records**: Verify your bank or credit card statement to confirm that two separate payments were processed.

3. **Contact Customer Service**: Reach out to the company’s billing or customer service department. Have your invoice number, account details, and any proof of double payment ready to provide for reference.

4. **Be Clear and Concise**: When contacting them, explain the situation clearly. Mention the invoice number, the date of the charges, the amount that was charged twice, and any other relevant details.

5. **Request a Refund or Correction**: Politely request a refund for the dupl

In [9]:
# Streaming

output = stream_invoke(router, {"input": "I was charged twice on my last invoice"})
output

────────────────────────────────────────────────────────────────────────────────

  STREAMING EXECUTION

────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────

  EXECUTION COMPLETE

────────────────────────────────────────────────────────────────────────────────

{'input': 'I was charged twice on my last invoice',
 'category': 'billing',
 'response': "If you were charged twice on your last invoice, you’ll want to address this issue promptly to ensure a refund or adjustment. Here are the steps you can take to resolve the double charge:\n\n1. **Review the Invoice**: Double-check the invoice details to confirm that there are indeed two charges for the same item or service, and ensure it’s not a misunderstanding or legitimate charge.\n\n2. **Check Payment Records**: Verify your bank or credit card statement to confirm that two separate payments were processed.\n\n3. **Contact Customer Service**: Reach out to the company’s billing or customer service department. Have your invoice number, account details, and any proof of double payment ready to provide for reference.\n\n4. **Be Clear and Concise**: When contacting them, explain the situation clearly. Mention the invoice number, the date of the charges, the amount that was charged twice, and any othe

In [10]:
print(output['response'])

If you were charged twice on your last invoice, you’ll want to address this issue promptly to ensure a refund or adjustment. Here are the steps you can take to resolve the double charge:

1. **Review the Invoice**: Double-check the invoice details to confirm that there are indeed two charges for the same item or service, and ensure it’s not a misunderstanding or legitimate charge.

2. **Check Payment Records**: Verify your bank or credit card statement to confirm that two separate payments were processed.

3. **Contact Customer Service**: Reach out to the company’s billing or customer service department. Have your invoice number, account details, and any proof of double payment ready to provide for reference.

4. **Be Clear and Concise**: When contacting them, explain the situation clearly. Mention the invoice number, the date of the charges, the amount that was charged twice, and any other relevant details.

5. **Request a Refund or Correction**: Politely request a refund for the dupl